In [3]:
import pandas as pd
import numpy as np



In [7]:
df = pd.read_csv("../outputs/yolov8n/metrics/yolov8n_actual_metrics.csv")

df

,video_name,zone_1,zone_2,zone_3,zone_4,zone_5,zone_6,Actual_people,actual_zone_1,actual_zone_2,...,actual_zone_6,model_name,frame_number,processed_frame,timestamp_sec,detected_people_count,unassigned_people,total_zones,occupied_zones,inference_time_ms
0,library_0,0,1,1,1,NaN,NaN,12,4,4,...,NaN,yolov8n,6,1,0.2,12,0,4,3,275.61
1,library_0,0,1,1,1,NaN,NaN,12,4,4,...,NaN,yolov8n,54,9,1.8,11,1,4,3,65.99
2,library_0,1,1,1,1,NaN,NaN,14,5,4,...,NaN,yolov8n,108,18,3.6,14,1,4,4,53.88
3,library_0,0,1,1,1,NaN,NaN,14,5,4,...,NaN,yolov8n,162,27,5.4,14,4,4,3,65.06
4,library_0,0,1,1,1,NaN,NaN,16,5,4,...,NaN,yolov8n,210,35,7.0,14,3,4,3,62.28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,office_0,0,1,1,0,NaN,NaN,4,1,2,...,NaN,yolov8n,1146,191,38.2,2,0,4,2,47.77
156,office_0,0,1,1,0,NaN,NaN,4,1,2,...,NaN,yolov8n,1218,203,40.6,2,0,4,2,57.54
157,office_0,0,1,0,0,NaN,NaN,4,0,1,...,NaN,yolov8n,1296,216,43.2,2,0,4,1,57.99
158,office_0,0,1,0,0,NaN,NaN,4,1,3,...,NaN,yolov8n,1374,229,45.8,3,0,4,1,58.92


In [8]:
def detection_accuracy(actual, detected):

    if actual == 0:
        return 100 if detected == 0 else 0

    acc = (1 - abs(actual - detected) / actual) * 100

    return max(acc, 0)

df["detection_accuracy"] = df.apply(
    lambda row: detection_accuracy(
        row["Actual_people"],
        row["detected_people_count"]
    ),
    axis=1
)

df[["video_name","frame_number","detection_accuracy"]].head()

video_detection_accuracy = (
    df.groupby("video_name")["detection_accuracy"]
      .mean()
      .reset_index()
      .rename(columns={
          "detection_accuracy":"Detection_Accuracy_%"
      })
)

video_detection_accuracy


,video_name,Detection_Accuracy_%
0,library_0,92.041667
1,market_0,93.333333
2,market_1,91.083333
3,market_2,83.770979
4,office_0,46.666667
5,test_0,89.599026
6,test_1,76.995671
7,test_2,84.855865


In [9]:
overall_detection_accuracy = (
    df["detection_accuracy"].mean()
)

print(
    f"Overall Detection Accuracy: "
    f"{overall_detection_accuracy:.2f}%"
)

Overall Detection Accuracy: 82.29%


6. Zone assignment accuracy

In [10]:

def zone_accuracy(actual, predicted):

    if pd.isna(actual):
        return np.nan

    if actual == 0:
        return 100 if predicted == 0 else 0

    acc = (1 - abs(actual - predicted) / actual) * 100

    return max(acc, 0)

In [11]:
# zone accuracy for each frame
zone_accuracy_cols = []

for zone in range(1, 7):

    actual_col = f"actual_zone_{zone}"
    pred_col = f"zone_{zone}"

    acc_col = f"zone_{zone}_accuracy"

    df[acc_col] = df.apply(
        lambda row: zone_accuracy(
            row[actual_col],
            row[pred_col]
        ),
        axis=1
    )

    zone_accuracy_cols.append(acc_col)

df.head()

,video_name,zone_1,zone_2,zone_3,zone_4,zone_5,zone_6,Actual_people,actual_zone_1,actual_zone_2,...,total_zones,occupied_zones,inference_time_ms,detection_accuracy,zone_1_accuracy,zone_2_accuracy,zone_3_accuracy,zone_4_accuracy,zone_5_accuracy,zone_6_accuracy
0,library_0,0,1,1,1,NaN,NaN,12,4,4,...,4,3,275.61,100.000000,0.0,25.0,33.333333,25.0,NaN,NaN
1,library_0,0,1,1,1,NaN,NaN,12,4,4,...,4,3,65.99,91.666667,0.0,25.0,33.333333,25.0,NaN,NaN
2,library_0,1,1,1,1,NaN,NaN,14,5,4,...,4,4,53.88,100.000000,20.0,25.0,33.333333,25.0,NaN,NaN
3,library_0,0,1,1,1,NaN,NaN,14,5,4,...,4,3,65.06,100.000000,0.0,25.0,33.333333,25.0,NaN,NaN
4,library_0,0,1,1,1,NaN,NaN,16,5,4,...,4,3,62.28,87.500000,0.0,25.0,33.333333,25.0,NaN,NaN


In [12]:
# frame wise zone wise assignement
df["zone_assignment_accuracy"] = (
    df[zone_accuracy_cols]
      .mean(axis=1, skipna=True)
)

df[
    [
        "video_name",
        "frame_number",
        "zone_assignment_accuracy"
    ]
].head()

,video_name,frame_number,zone_assignment_accuracy
0,library_0,6,20.833333
1,library_0,54,20.833333
2,library_0,108,25.833333
3,library_0,162,20.833333
4,library_0,210,20.833333


In [13]:
# zone assignement per video
video_zone_accuracy = (
    df.groupby("video_name")
      ["zone_assignment_accuracy"]
      .mean()
      .reset_index()
      .rename(columns={
          "zone_assignment_accuracy":
          "Zone_Assignment_Accuracy_%"
      })
)

video_zone_accuracy

,video_name,Zone_Assignment_Accuracy_%
0,library_0,44.770833
1,market_0,92.500000
2,market_1,100.000000
3,market_2,74.750000
4,office_0,37.708333
5,test_0,47.194444
6,test_1,21.309524
7,test_2,100.000000


In [14]:
# overall zone assignement accuracy
overall_zone_accuracy = (
    df["zone_assignment_accuracy"]
      .mean()
)

print(
    f"Overall Zone Assignment Accuracy: "
    f"{overall_zone_accuracy:.2f}%"
)

Overall Zone Assignment Accuracy: 64.78%


In [15]:
final_metrics = (
    video_detection_accuracy
    .merge(
        video_zone_accuracy,
        on="video_name"
    )
)

final_metrics

,video_name,Detection_Accuracy_%,Zone_Assignment_Accuracy_%
0,library_0,92.041667,44.770833
1,market_0,93.333333,92.500000
2,market_1,91.083333,100.000000
3,market_2,83.770979,74.750000
4,office_0,46.666667,37.708333
5,test_0,89.599026,47.194444
6,test_1,76.995671,21.309524
7,test_2,84.855865,100.000000


# s yolov8s

In [21]:
df = pd.read_csv("../outputs/yolov8s/metrics/yolov8s_actual_metrics.csv")

df

,video_name,zone_1,zone_2,zone_3,zone_4,zone_5,zone_6,Actual_people,actual_zone_1,actual_zone_2,...,actual_zone_6,model_name,frame_number,processed_frame,timestamp_sec,detected_people_count,unassigned_people,total_zones,occupied_zones,inference_time_ms
0,library_0,0,1,1,1,NaN,NaN,12,4,4,...,NaN,yolov8s,6,1,0.2,14,1,4,3,475.24
1,library_0,1,1,1,1,NaN,NaN,12,4,4,...,NaN,yolov8s,54,9,1.8,14,1,4,4,193.20
2,library_0,1,1,1,1,NaN,NaN,14,5,4,...,NaN,yolov8s,108,18,3.6,13,1,4,4,196.99
3,library_0,0,1,1,1,NaN,NaN,14,5,4,...,NaN,yolov8s,162,27,5.4,14,2,4,3,219.56
4,library_0,0,1,1,1,NaN,NaN,14,5,4,...,NaN,yolov8s,210,35,7.0,13,2,4,3,179.37
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,office_0,1,1,1,0,NaN,NaN,4,1,2,...,NaN,yolo11n,1146,191,38.2,3,0,4,3,106.06
316,office_0,1,1,1,0,NaN,NaN,4,1,2,...,NaN,yolo11n,1218,203,40.6,4,0,4,3,96.29
317,office_0,1,1,1,0,NaN,NaN,5,0,1,...,NaN,yolo11n,1296,216,43.2,5,0,4,3,107.06
318,office_0,1,1,1,0,NaN,NaN,5,1,3,...,NaN,yolo11n,1374,229,45.8,5,0,4,3,96.42


In [22]:
def detection_accuracy(actual, detected):

    if actual == 0:
        return 100 if detected == 0 else 0

    acc = (1 - abs(actual - detected) / actual) * 100

    return max(acc, 0)

df["detection_accuracy"] = df.apply(
    lambda row: detection_accuracy(
        row["Actual_people"],
        row["detected_people_count"]
    ),
    axis=1
)

df[["video_name","frame_number","detection_accuracy"]].head()

video_detection_accuracy = (
    df.groupby("video_name")["detection_accuracy"]
      .mean()
      .reset_index()
      .rename(columns={
          "detection_accuracy":"Detection_Accuracy_%"
      })
)

print(video_detection_accuracy)

overall_detection_accuracy = (
    df["detection_accuracy"].mean()
)

print(
    f"Overall Detection Accuracy: "
    f"{overall_detection_accuracy:.2f}%"
)


  video_name  Detection_Accuracy_%
0  library_0             76.392857
1   market_0             80.952381
2   market_1             68.625000
3   market_2             82.970862
4   office_0             65.416667
5     test_0             87.523722
6     test_1             79.513889
7     test_2             89.726190
Overall Detection Accuracy: 78.89%


6. Zone assignment accuracy

In [23]:

def zone_accuracy(actual, predicted):

    if pd.isna(actual):
        return np.nan

    if actual == 0:
        return 100 if predicted == 0 else 0

    acc = (1 - abs(actual - predicted) / actual) * 100

    return max(acc, 0)

# zone accuracy for each frame
zone_accuracy_cols = []

for zone in range(1, 7):

    actual_col = f"actual_zone_{zone}"
    pred_col = f"zone_{zone}"

    acc_col = f"zone_{zone}_accuracy"

    df[acc_col] = df.apply(
        lambda row: zone_accuracy(
            row[actual_col],
            row[pred_col]
        ),
        axis=1
    )

    zone_accuracy_cols.append(acc_col)

# df.head()

# frame wise zone wise assignement
df["zone_assignment_accuracy"] = (
    df[zone_accuracy_cols]
      .mean(axis=1, skipna=True)
)

df[
    [
        "video_name",
        "frame_number",
        "zone_assignment_accuracy"
    ]
].head()

,video_name,frame_number,zone_assignment_accuracy
0,library_0,6,20.833333
1,library_0,54,27.083333
2,library_0,108,25.833333
3,library_0,162,20.833333
4,library_0,210,20.833333


In [24]:
# zone assignement per video
video_zone_accuracy = (
    df.groupby("video_name")
      ["zone_assignment_accuracy"]
      .mean()
      .reset_index()
      .rename(columns={
          "zone_assignment_accuracy":
          "Zone_Assignment_Accuracy_%"
      })
)

print(video_zone_accuracy)

# overall zone assignement accuracy
overall_zone_accuracy = (
    df["zone_assignment_accuracy"]
      .mean()
)

print(
    f"Overall Zone Assignment Accuracy: "
    f"{overall_zone_accuracy:.2f}%"
)

  video_name  Zone_Assignment_Accuracy_%
0  library_0                   44.468750
1   market_0                   91.666667
2   market_1                   97.500000
3   market_2                   70.750000
4   office_0                   48.645833
5     test_0                   47.194444
6     test_1                   18.294643
7     test_2                  100.000000
Overall Zone Assignment Accuracy: 64.82%


In [25]:
final_metrics = (
    video_detection_accuracy
    .merge(
        video_zone_accuracy,
        on="video_name"
    )
)

final_metrics

,video_name,Detection_Accuracy_%,Zone_Assignment_Accuracy_%
0,library_0,76.392857,44.468750
1,market_0,80.952381,91.666667
2,market_1,68.625000,97.500000
3,market_2,82.970862,70.750000
4,office_0,65.416667,48.645833
5,test_0,87.523722,47.194444
6,test_1,79.513889,18.294643
7,test_2,89.726190,100.000000


# 11 yolo11n

In [26]:
df = pd.read_csv("../outputs/yolo11n/metrics/yolo11n_actual_metrics.csv")

df

,video_name,zone_1,zone_2,zone_3,zone_4,zone_5,zone_6,Actual_people,actual_zone_1,actual_zone_2,...,actual_zone_6,model_name,frame_number,processed_frame,timestamp_sec,detected_people_count,unassigned_people,total_zones,occupied_zones,inference_time_ms
0,library_0,0,1,1,1,NaN,NaN,12,4,4,...,NaN,yolo11n,6,1,0.2,9,0,4,3,313.92
1,library_0,0,1,0,1,NaN,NaN,12,4,4,...,NaN,yolo11n,54,9,1.8,10,1,4,2,211.72
2,library_0,1,1,1,1,NaN,NaN,14,5,4,...,NaN,yolo11n,108,18,3.6,11,1,4,4,112.47
3,library_0,1,1,0,1,NaN,NaN,14,5,4,...,NaN,yolo11n,162,27,5.4,12,3,4,3,96.38
4,library_0,0,1,1,1,NaN,NaN,14,5,4,...,NaN,yolo11n,210,35,7.0,12,2,4,3,117.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,office_0,1,1,1,0,NaN,NaN,4,1,2,...,NaN,yolo11n,1146,191,38.2,3,0,4,3,106.06
156,office_0,1,1,1,0,NaN,NaN,4,1,2,...,NaN,yolo11n,1218,203,40.6,4,0,4,3,96.29
157,office_0,1,1,1,0,NaN,NaN,5,0,1,...,NaN,yolo11n,1296,216,43.2,5,0,4,3,107.06
158,office_0,1,1,1,0,NaN,NaN,5,1,3,...,NaN,yolo11n,1374,229,45.8,5,0,4,3,96.42


In [27]:
def detection_accuracy(actual, detected):

    if actual == 0:
        return 100 if detected == 0 else 0

    acc = (1 - abs(actual - detected) / actual) * 100

    return max(acc, 0)

df["detection_accuracy"] = df.apply(
    lambda row: detection_accuracy(
        row["Actual_people"],
        row["detected_people_count"]
    ),
    axis=1
)

df[["video_name","frame_number","detection_accuracy"]].head()

video_detection_accuracy = (
    df.groupby("video_name")["detection_accuracy"]
      .mean()
      .reset_index()
      .rename(columns={
          "detection_accuracy":"Detection_Accuracy_%"
      })
)

print(video_detection_accuracy)

overall_detection_accuracy = (
    df["detection_accuracy"].mean()
)

print(
    f"Overall Detection Accuracy: "
    f"{overall_detection_accuracy:.2f}%"
)


  video_name  Detection_Accuracy_%
0  library_0             76.595238
1   market_0             81.904762
2   market_1             62.416667
3   market_2             84.876457
4   office_0             55.000000
5     test_0             84.627880
6     test_1             79.500000
7     test_2             86.547619
Overall Detection Accuracy: 76.43%


6. Zone assignment accuracy

In [28]:

def zone_accuracy(actual, predicted):

    if pd.isna(actual):
        return np.nan

    if actual == 0:
        return 100 if predicted == 0 else 0

    acc = (1 - abs(actual - predicted) / actual) * 100

    return max(acc, 0)

# zone accuracy for each frame
zone_accuracy_cols = []

for zone in range(1, 7):

    actual_col = f"actual_zone_{zone}"
    pred_col = f"zone_{zone}"

    acc_col = f"zone_{zone}_accuracy"

    df[acc_col] = df.apply(
        lambda row: zone_accuracy(
            row[actual_col],
            row[pred_col]
        ),
        axis=1
    )

    zone_accuracy_cols.append(acc_col)

# df.head()

# frame wise zone wise assignement
df["zone_assignment_accuracy"] = (
    df[zone_accuracy_cols]
      .mean(axis=1, skipna=True)
)

df[
    [
        "video_name",
        "frame_number",
        "zone_assignment_accuracy"
    ]
].head()

,video_name,frame_number,zone_assignment_accuracy
0,library_0,6,20.833333
1,library_0,54,12.500000
2,library_0,108,25.833333
3,library_0,162,17.500000
4,library_0,210,20.833333


In [29]:
# zone assignement per video
video_zone_accuracy = (
    df.groupby("video_name")
      ["zone_assignment_accuracy"]
      .mean()
      .reset_index()
      .rename(columns={
          "zone_assignment_accuracy":
          "Zone_Assignment_Accuracy_%"
      })
)

print(video_zone_accuracy)

# overall zone assignement accuracy
overall_zone_accuracy = (
    df["zone_assignment_accuracy"]
      .mean()
)

print(
    f"Overall Zone Assignment Accuracy: "
    f"{overall_zone_accuracy:.2f}%"
)

  video_name  Zone_Assignment_Accuracy_%
0  library_0                   43.145833
1   market_0                   90.000000
2   market_1                   96.666667
3   market_2                   71.750000
4   office_0                   41.458333
5     test_0                   47.194444
6     test_1                   13.125000
7     test_2                  100.000000
Overall Zone Assignment Accuracy: 62.92%


In [30]:
final_metrics = (
    video_detection_accuracy
    .merge(
        video_zone_accuracy,
        on="video_name"
    )
)

final_metrics

,video_name,Detection_Accuracy_%,Zone_Assignment_Accuracy_%
0,library_0,76.595238,43.145833
1,market_0,81.904762,90.000000
2,market_1,62.416667,96.666667
3,market_2,84.876457,71.750000
4,office_0,55.000000,41.458333
5,test_0,84.627880,47.194444
6,test_1,79.500000,13.125000
7,test_2,86.547619,100.000000
